In [ ]:
############################################################
# 11. RetinaNet 모델 정의 + 학습
############################################################

from torchvision.models.detection import retinanet_resnet50_fpn
from torchvision.models.detection.retinanet import RetinaNetClassificationHead

def build_retinanet_model(num_classes):
    retina_model = retinanet_resnet50_fpn(weights="DEFAULT")
    cls_head = retina_model.head.classification_head

    # torchvision 버전에 따라 conv[0]이 Conv2dNormActivation일 수 있으므로
    # 내부 첫 Conv2d를 찾아 채널 수를 읽는다.
    first_conv = next(
        module for module in cls_head.conv.modules() if isinstance(module, nn.Conv2d)
    )
    in_channels = first_conv.in_channels
    num_anchors = cls_head.num_anchors

    retina_model.head.classification_head = RetinaNetClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=num_classes,
    )
    return retina_model


# 사전학습된 RetinaNet 모델 로드
retina_model = build_retinanet_model(num_classes)
retina_model.to(DEVICE)

# 옵티마이저 / 스케줄러 설정
retina_optimizer = optim.AdamW(
    [p for p in retina_model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=1e-4,
)
retina_scheduler = optim.lr_scheduler.StepLR(retina_optimizer, step_size=3, gamma=0.1)


############################################################
# 12. RetinaNet 학습 루프
############################################################

retina_num_epochs = 5

for epoch in range(retina_num_epochs):
    ##########################
    # 1) Train
    ##########################
    retina_model.train()
    retina_train_loss_sum = 0.0

    for images, targets in train_loader:
        images = [img.to(DEVICE) for img in images]
        targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

        loss_dict = retina_model(images, targets)
        losses = sum(loss for loss in loss_dict.values())

        retina_optimizer.zero_grad()
        losses.backward()
        retina_optimizer.step()

        retina_train_loss_sum += losses.item()

    avg_retina_train_loss = retina_train_loss_sum / max(1, len(train_loader))

    ##########################
    # 2) Validation (loss만 측정)
    ##########################
    retina_model.train()  # detection 모델은 train() 상태에서 loss를 반환
    retina_val_loss_sum = 0.0
    with torch.no_grad():
        for images, targets in val_loader:
            images = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]

            loss_dict = retina_model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            retina_val_loss_sum += losses.item()

    avg_retina_val_loss = retina_val_loss_sum / max(1, len(val_loader))

    print(
        f"[RetinaNet Epoch {epoch+1}/{retina_num_epochs}] "
        f"train_loss: {avg_retina_train_loss:.4f} | "
        f"val_loss: {avg_retina_val_loss:.4f}"
    )

    retina_scheduler.step()

# 학습된 모델 저장
torch.save(retina_model.state_dict(), "retinanet_oral_drug.pth")
print("RetinaNet 모델 저장 완료")